# Reel Brain Report

See how an average brain is **predicted** to respond to your reel, second by second, using Meta FAIR's
[TRIBE v2](https://github.com/facebookresearch/tribev2). This notebook runs everything on Google Colab's
free GPU and gives you a `report.zip` to open in the [report viewer](https://reel-brain-analysis.vercel.app).

**Before you start (one time only)**
1. A [Hugging Face](https://huggingface.co) account. Request access to
   [meta-llama/Llama-3.2-3B](https://huggingface.co/meta-llama/Llama-3.2-3B) and wait for the approval email
   (minutes to a few hours).
2. A **Read** token from https://huggingface.co/settings/tokens. In Colab, click the 🔑 key icon on the left →
   **Add new secret** → name `HF_TOKEN`, paste the token as the value, and switch on **Notebook access**.

**Each time**
1. `Runtime` → `Change runtime type` → **T4 GPU** → Save.
2. Run the cells **one at a time, top to bottom** (click ▶ and wait for each to finish).

Reels must be **60 seconds or shorter**. Expect roughly 15–40 minutes on the free T4, depending on length.

*TRIBE v2 is licensed CC BY-NC 4.0 (non-commercial use only); Llama 3.2 is under the Llama 3.2 Community
License. By running this you agree to both.*

## 1. Check the GPU
You should see a **Tesla T4** (or another NVIDIA GPU). If you get an error, set the runtime to T4 GPU (see above).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Install (about 5–10 minutes)

In [ ]:
import subprocess

def install(name, cmd):
    print(f"Installing {name}...")
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode:
        print(result.stdout[-3000:] + result.stderr[-3000:])
        raise SystemExit(f"Installing {name} failed.")

install("TRIBE v2", "([ -d /opt/tribev2 ] || git clone -q https://github.com/facebookresearch/tribev2 /opt/tribev2)"
        " && cd /opt/tribev2 && git checkout -q af58661791a351a448a489042a28f6c37e1c14b7 && pip install -e .")
install("PyTorch 2.6", "pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0"
        " --index-url https://download.pytorch.org/whl/cu124")
install("the report builder", "pip install nilearn==0.13.1"
        " && pip install --no-deps 'git+https://github.com/ammarahmad1/reel-brain-analysis.git#subdirectory=packages/report'")
install("the English language model", "python -m spacy download en_core_web_lg")
install("the speech tokenizer", "mkdir -p /root/nltk_data/tokenizers"
        " && wget -q -O /tmp/punkt_tab.zip https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages/tokenizers/punkt_tab.zip"
        " && unzip -q -o /tmp/punkt_tab.zip -d /root/nltk_data/tokenizers/")
print("Done.")

## 3. Sign in to Hugging Face
Uses the `HF_TOKEN` secret you added (see the top of this notebook).

In [ ]:
import os
from google.colab import userdata
from huggingface_hub import login

try:
    token = userdata.get("HF_TOKEN")
except Exception:
    token = None
if not token:
    raise SystemExit(
        "No HF_TOKEN found. Click the key icon on the left, add a secret named HF_TOKEN with your "
        "Hugging Face token, switch on Notebook access, then run this cell again."
    )
os.environ["HF_TOKEN"] = token
login(token=token)
print("Signed in.")

## 4. Choose your reel
One video, **60 seconds or shorter** (MP4, MOV or WebM), that you made or have the rights to use.

1. Run the cell below.
2. A **Choose Files** button appears under the cell. Click it and pick your reel.
3. Wait for the upload to finish. You'll see **Ready:** with your reel's name and length.

**Upload stuck or slow?** Use the file panel instead:
1. Click the 📁 folder icon on the left and drag your reel into the panel. Wait until it appears in the list.
2. Right-click the reel → **Copy path**, then paste it between the quotes in `EXISTING_VIDEO = ""` below,
   so it reads like `EXISTING_VIDEO = "/content/my_reel.mp4"`.
3. Run the cell.

In [ ]:
import shutil, subprocess
from pathlib import Path
from google.colab import files

EXISTING_VIDEO = ""
MAX_SECONDS = 60
VIDEO_EXTS = {".mp4", ".mov", ".m4v", ".webm", ".mkv", ".avi"}

reel_dir = Path("/content/reel")
shutil.rmtree(reel_dir, ignore_errors=True)
reel_dir.mkdir()
if EXISTING_VIDEO:
    shutil.copy(EXISTING_VIDEO, reel_dir)
else:
    for name, data in files.upload().items():
        (reel_dir / Path(name).name).write_bytes(data)
        Path(name).unlink(missing_ok=True)

videos = [p for p in reel_dir.iterdir() if p.suffix.lower() in VIDEO_EXTS]
if len(videos) != 1:
    raise SystemExit("Please choose exactly one video file (MP4, MOV or WebM).")
video = videos[0]
duration = float(subprocess.run(
    ["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "csv=p=0", str(video)],
    capture_output=True, text=True, check=True).stdout)
if duration > MAX_SECONDS + 0.5:
    raise SystemExit(f"This reel is {duration:.0f}s long. Please trim it to {MAX_SECONDS} seconds or less.")
print(f"Ready: {video.name} ({duration:.1f}s)")

## 5. Run TRIBE v2 and build the report
The first run downloads about 15 GB of model weights, then analyses your reel. This is the long step:
keep this tab open. You'll see warnings scroll past; they're normal.

In [ ]:
!rm -rf /content/report && python -u -m reel_report "{video}" --out /content/report

## 6. Download your report
Downloads `report.zip`. Open the [report viewer](https://reel-brain-analysis.vercel.app) and drop the zip onto it to see the full report.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("/content/report", "zip", "/content/report")
files.download("/content/report.zip")

---
Optional: delete your reel and the extracted audio from this Colab session (Colab also wipes everything when the
session ends).

In [ ]:
import shutil
shutil.rmtree("/content/reel", ignore_errors=True)
print("Deleted.")